# Model 2: Xception — Siamese Multimodal Network

Notebook này có cấu trúc **giống hệt** `03_model_inceptionresnet.ipynb`,
chỉ thay backbone thành **Xception**.

**Xception** (Depthwise Separable Convolutions): được chứng minh có khả năng
chống overfitting tốt trên tập validation (Accuracy 86.60%, AUC 0.8435 trong
nghiên cứu Multi-Disease Retinal Classification).

**Kiến trúc**: Giống notebook 03 với IMAGE_SIZE=299, BATCH_SIZE=24

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ===== CẤU HÌNH =====
PREPROC_DIR  = f'/kaggle/input/datasets/xuanductran/odir-data-preprocess-augmentation/odir-data-preprocess-augmentation'
MODEL_DIR    = f'/kaggle/working'
CHECKPT_DIR  = f'{MODEL_DIR}/checkpoints'
HISTORY_DIR  = f'{MODEL_DIR}/history'
os.makedirs(CHECKPT_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

TARGET_COLS   = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
BACKBONE_TAG  = 'xception'
IMAGE_SIZE    = 299
BATCH_SIZE    = 24
NUM_EPOCHS    = 50
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
DROPOUT       = 0.4
PROJ_DIM      = 128
TABULAR_DIM   = 16
NUM_WORKERS   = 4
PATIENCE      = 8

# Xác minh tên backbone trong timm
available_xception = timm.list_models('*xception*', pretrained=True)
print(f'Available xception models: {available_xception}')
# Chọn 'xception' nếu có, ngược lại dùng 'legacy_xception'
if 'xception' in available_xception:
    BACKBONE = 'xception'
elif 'legacy_xception' in available_xception:
    BACKBONE = 'legacy_xception'
elif available_xception:
    BACKBONE = available_xception[0]
else:
    raise ValueError('Không tìm thấy Xception trong timm. Cài: pip install timm')

print(f'Sử dụng backbone: {BACKBONE}')

## 1. Load dữ liệu

In [ ]:
meta_df = pd.read_csv(f'{PREPROC_DIR}/split_metadata.csv')
OLD_PREFIX = '/home/centrala/work/ou/kltn/Ocular-Disease-Recognition/odir-data-preprocess-augmentation'
meta_df['left_path']  = meta_df['left_path'].str.replace(OLD_PREFIX, PREPROC_DIR, regex=False)
meta_df['right_path'] = meta_df['right_path'].str.replace(OLD_PREFIX, PREPROC_DIR, regex=False)

with open(f'{PREPROC_DIR}/class_weights.json') as f:
    class_weights_dict = json.load(f)
with open(f'{MODEL_DIR}/norm_params.json') as f:
    norm_params = json.load(f)

AGE_MEAN = norm_params['age_mean']
AGE_STD  = norm_params['age_std']

pos_weight = torch.tensor(
    [class_weights_dict[c] for c in TARGET_COLS],
    dtype=torch.float32
).to(device)

train_df = meta_df[meta_df['split'] == 'train'].reset_index(drop=True)
val_df   = meta_df[(meta_df['split'] == 'val') & (meta_df['aug_id'] == -1)].reset_index(drop=True)

print(f'Train rows: {len(train_df)} | Val rows: {len(val_df)}')
print(f'Age normalization — mean: {AGE_MEAN:.2f}, std: {AGE_STD:.2f}')

## 2. Dataset và DataLoader

In [ ]:
def encode_gender(gender_str):
    s = str(gender_str).strip().lower()
    return [1.0, 0.0] if s in ('male', 'm') else [0.0, 1.0]


class OdirDataset(Dataset):
    def __init__(self, df, target_cols, image_size, age_mean, age_std):
        self.df          = df.reset_index(drop=True)
        self.target_cols = target_cols
        self.age_mean    = age_mean
        self.age_std     = age_std
        self.transform   = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        left_t    = self.transform(Image.open(row['left_path']).convert('RGB'))
        right_t   = self.transform(Image.open(row['right_path']).convert('RGB'))
        age_norm  = (row['age'] - self.age_mean) / (self.age_std + 1e-8)
        tabular   = torch.tensor([age_norm] + encode_gender(row['gender']), dtype=torch.float32)
        labels    = torch.tensor([float(row[c]) for c in self.target_cols], dtype=torch.float32)
        return left_t, right_t, tabular, labels


train_loader = DataLoader(
    OdirDataset(train_df, TARGET_COLS, IMAGE_SIZE, AGE_MEAN, AGE_STD),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    OdirDataset(val_df, TARGET_COLS, IMAGE_SIZE, AGE_MEAN, AGE_STD),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 3. Kiến trúc mô hình

In [ ]:
class SiameseMultimodalNet(nn.Module):
    def __init__(self, backbone_name, proj_dim=128, tabular_dim=16,
                 dropout=0.4, num_classes=8):
        super().__init__()
        self.backbone       = timm.create_model(backbone_name, pretrained=True,
                                                num_classes=0, global_pool='avg')
        feat_dim            = self.backbone.num_features
        self.projector      = nn.Sequential(
            nn.Linear(feat_dim, proj_dim), nn.ReLU(inplace=True), nn.Dropout(dropout)
        )
        self.tabular_encoder = nn.Sequential(
            nn.Linear(3, tabular_dim), nn.ReLU(inplace=True)
        )
        self.classifier     = nn.Sequential(
            nn.Linear(2 * proj_dim + tabular_dim, 64),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward_one(self, x):
        return self.projector(self.backbone(x))

    def forward(self, left, right, tabular):
        img_feat = torch.cat([self.forward_one(left), self.forward_one(right)], dim=1)
        return self.classifier(torch.cat([img_feat, self.tabular_encoder(tabular)], dim=1))


model = SiameseMultimodalNet(
    backbone_name=BACKBONE, proj_dim=PROJ_DIM,
    tabular_dim=TABULAR_DIM, dropout=DROPOUT, num_classes=len(TARGET_COLS)
).to(device)

print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Backbone feat_dim: {model.backbone.num_features}')

## 4. Loss, Optimizer, Scheduler

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

## 5. Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for left, right, tabular, labels in loader:
            left, right, tabular, labels = (
                left.to(device), right.to(device),
                tabular.to(device), labels.to(device)
            )
            if train:
                optimizer.zero_grad()
            logits = model(left, right, tabular)
            loss   = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            all_logits.append(logits.detach().cpu())
            all_labels.append(labels.cpu())

    avg_loss  = total_loss / len(loader)
    all_probs = torch.sigmoid(torch.cat(all_logits)).numpy()
    all_labels = torch.cat(all_labels).numpy()
    try:
        auc = roc_auc_score(all_labels, all_probs, average='macro')
    except Exception:
        auc = 0.0
    return avg_loss, auc


history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': [], 'lr': []}
best_val_loss, patience_counter = float('inf'), 0
checkpoint_path = f'{CHECKPT_DIR}/{BACKBONE_TAG}_best.pth'

print(f'Training: {NUM_EPOCHS} epochs | Backbone: {BACKBONE}')
print('-' * 80)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_auc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    val_loss,   val_auc   = run_epoch(model, val_loader,   criterion, None,      device, train=False)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    history['lr'].append(current_lr)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train={train_loss:.4f}/{train_auc:.4f} | '
          f'Val={val_loss:.4f}/{val_auc:.4f} | '
          f'LR={current_lr:.2e} | {elapsed:.0f}s')

    if val_loss < best_val_loss:
        best_val_loss, patience_counter = val_loss, 0
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss, 'val_auc': val_auc,
            'backbone': BACKBONE,
            'config': {'image_size': IMAGE_SIZE, 'proj_dim': PROJ_DIM,
                       'tabular_dim': TABULAR_DIM, 'dropout': DROPOUT}
        }, checkpoint_path)
        print(f'  *** Best checkpoint (val_loss={val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping sau {epoch} epochs')
            break

print(f'Best val_loss: {best_val_loss:.4f}')

## 6. Lưu lịch sử và vẽ đồ thị

In [ ]:
history_path = f'{HISTORY_DIR}/{BACKBONE_TAG}_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

epochs = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Val')
ax1.set_title(f'Loss — {BACKBONE}'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, history['train_auc'], label='Train')
ax2.plot(epochs, history['val_auc'],   label='Val')
ax2.set_title(f'Macro AUC — {BACKBONE}'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/chart_{BACKBONE_TAG}_history.png', dpi=120)
plt.show()
print(f'Saved: {history_path}')
print('Bước tiếp theo: 05_model_resnet50.ipynb')